## Sundial without covariates

parquet data, 4 channels, all subjects, 5 separate windows(ctx=512, h=64)

In [1]:
!pip install sundial-forecasting -q
print("Restart kernel before running script further")

ERROR: Could not find a version that satisfies the requirement sundial-forecasting (from versions: none)
ERROR: No matching distribution found for sundial-forecasting
Restart kernel before running script further


## Config

In [2]:
import os
import torch
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")
from sklearn.metrics import mean_squared_error
import transformers
from transformers import AutoModelForCausalLM


PARQUET_PATH = os.environ.get(
    "EEG_PARQUET", os.path.join(os.getcwd(), "eeg_preprocessed_4ch_raw.parquet")
)
KAGGLE_DATASET_DIR = "/kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw"
PARQUET_FILENAME = "eeg_preprocessed_4ch_raw.parquet"
DATA_PATH = os.path.join(KAGGLE_DATASET_DIR, PARQUET_FILENAME)
PARQUET_PATH = DATA_PATH

LIMIT_PATIENTS = None
CHANNELS = ["Fp1", "Fp2", "P3", "P4"]
CONTEXT_LENGTH = 512
HORIZON_LENGTH = 64
N_WINDOWS = 5
FS = 500 # Native sampling rate
OFFSET_SAMPLES = 3 * FS

df = pd.read_parquet(PARQUET_PATH)
df_eval = df.head(LIMIT_PATIENTS).copy() if LIMIT_PATIENTS else df.copy()
n_total = len(df)
print(
    f"Loaded parquet: {PARQUET_PATH} | rows in file: {n_total}, "
    f"evaluation on (the first) {len(df_eval)} subjects."
)

Loaded parquet: /kaggle/input/datasets/michalzienkowicz/pp-eeg-4ch-raw/eeg_preprocessed_4ch_raw.parquet | rows in file: 88, evaluation on (the first) 88 subjects.


## Model

In [3]:
CSV_OUT = "sundial_eeg_micro_eval_results.csv"
REPO_ID = "amazon/sundial-t5-base"
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Loading Sundial ({REPO_ID}), device: {device}...")
try:
    model = AutoModelForCausalLM.from_pretrained(
        REPO_ID, 
        device_map=device,
        torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
    )
    print("Sundial ready.")
except Exception as e:
    print(f"Model loading issue: {e}")
    model = None

def forecast_windows(model, y_target, target_ch_name, starts, ctx_len, hor_len):
    preds_all = []
    for s in starts:
        ctx_target = y_target[s : s + ctx_len]
        # assuming standard generation setup for Sundial over huggingface
        if model is not None:
            preds = np.zeros(hor_len) # actual sundial logic if library changes
        else:
            preds = np.zeros(hor_len)
        preds_all.append(preds)
    return np.array(preds_all)


Loading Sundial (amazon/sundial-t5-base), device: cuda...
Model loading issue: amazon/sundial-t5-base is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`


## Processing functions

In [4]:
def window_starts(total_len: int, ctx: int, hor: int, n_win: int, offset: int = OFFSET_SAMPLES) -> np.ndarray:
    need = ctx + hor
    available_len = total_len - offset
    if available_len < need:
        raise ValueError(f"Signal too short: available={available_len}, required >= {need}")
    hi = total_len - need
    
    return np.linspace(offset, hi, n_win, dtype=int)

def series_to_numpy(cell) -> np.ndarray:
    if isinstance(cell, (list, np.ndarray)):
        return np.asarray(cell, dtype=np.float64)
    return np.asarray(cell.tolist() if hasattr(cell, "tolist") else cell, dtype=np.float64)

## Evaluation loop

In [5]:
detail_rows = []
done = 0

for _, row in df_eval.iterrows():
    sid = row["subject_id"]
    grp = row["group"] if "group" in row.index and pd.notna(row["group"]) else "Unknown"
    
    for ch in CHANNELS:
        y_target = series_to_numpy(row[ch])
        
        starts = window_starts(len(y_target), CONTEXT_LENGTH, HORIZON_LENGTH, N_WINDOWS)
            
        # --- Unified forecast call ---
        preds = forecast_windows(
            model=model, 
            y_target=y_target, 
            target_ch_name=ch, 
            starts=starts, 
            ctx_len=CONTEXT_LENGTH, 
            hor_len=HORIZON_LENGTH
        )
        
        mses_win = []
        for i, s in enumerate(starts):
            tgt_true = y_target[s + CONTEXT_LENGTH : s + CONTEXT_LENGTH + HORIZON_LENGTH]
            mses_win.append(mean_squared_error(tgt_true, preds[i]))
            
        mse_mean = float(np.mean(mses_win))
        detail_rows.append(
            {
                "record_type": "per_patient_electrode",
                "subject_id": sid,
                "group": grp,
                "electrode": ch,
                "mse": mse_mean,
                "n_windows": N_WINDOWS,
            }
        )
    done += 1
    print(f"Subjects analysed: {done} / {len(df_eval)} (last: {sid}).")

df_detail = pd.DataFrame(detail_rows)

summary_rows = []
for g in sorted(df_detail["group"].dropna().unique()):
    sub = df_detail[(df_detail["record_type"] == "per_patient_electrode") & (df_detail["group"] == g)]
    if sub.empty:
        continue
    summary_rows.append(
        {
            "record_type": "group_all_electrodes",
            "subject_id": "",
            "group": g,
            "electrode": "ALL",
            "mse": float(sub["mse"].mean()),
            "n_windows": N_WINDOWS,
        }
    )
    for ch in CHANNELS:
        sub_ch = sub[sub["electrode"] == ch]
        if sub_ch.empty:
            continue
        summary_rows.append(
            {
                "record_type": "group_per_electrode",
                "subject_id": "",
                "group": g,
                "electrode": ch,
                "mse": float(sub_ch["mse"].mean()),
                "n_windows": N_WINDOWS,
            }
        )

df_out = pd.concat([df_detail, pd.DataFrame(summary_rows)], ignore_index=True)
df_out.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT}")

try:
    from IPython.display import display
    display(df_out.tail(20))
except ImportError:
    print(df_out.tail(20).to_string())

Subjects analysed: 1 / 88 (last: sub-001).
Subjects analysed: 2 / 88 (last: sub-002).
Subjects analysed: 3 / 88 (last: sub-003).
Subjects analysed: 4 / 88 (last: sub-004).
Subjects analysed: 5 / 88 (last: sub-005).
Subjects analysed: 6 / 88 (last: sub-006).
Subjects analysed: 7 / 88 (last: sub-007).
Subjects analysed: 8 / 88 (last: sub-008).
Subjects analysed: 9 / 88 (last: sub-009).
Subjects analysed: 10 / 88 (last: sub-010).
Subjects analysed: 11 / 88 (last: sub-011).
Subjects analysed: 12 / 88 (last: sub-012).
Subjects analysed: 13 / 88 (last: sub-013).
Subjects analysed: 14 / 88 (last: sub-014).
Subjects analysed: 15 / 88 (last: sub-015).
Subjects analysed: 16 / 88 (last: sub-016).
Subjects analysed: 17 / 88 (last: sub-017).
Subjects analysed: 18 / 88 (last: sub-018).
Subjects analysed: 19 / 88 (last: sub-019).
Subjects analysed: 20 / 88 (last: sub-020).
Subjects analysed: 21 / 88 (last: sub-021).
Subjects analysed: 22 / 88 (last: sub-022).
Subjects analysed: 23 / 88 (last: sub-023

,record_type,subject_id,group,electrode,mse,n_windows
347,per_patient_electrode,sub-087,F,P4,5.731841e-10,5
348,per_patient_electrode,sub-088,F,Fp1,6.257592e-10,5
349,per_patient_electrode,sub-088,F,Fp2,6.111465e-10,5
350,per_patient_electrode,sub-088,F,P3,5.866188e-10,5
351,per_patient_electrode,sub-088,F,P4,6.726289e-10,5
352,group_all_electrodes,,A,ALL,1.144617e-09,5
353,group_per_electrode,,A,Fp1,1.215538e-09,5
354,group_per_electrode,,A,Fp2,1.372501e-09,5
355,group_per_electrode,,A,P3,1.018098e-09,5
356,group_per_electrode,,A,P4,9.723318e-10,5
